# Notebook 47R1 - Advanced Restorer Trade-off Exploration
Adds Pareto overlays, outlier detection, per-source-group frontiers and page drill-down tables.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import mahalanobis
from scipy.spatial import ConvexHull

ROOT=Path("/content/drive/MyDrive/MusicAnnotationProject")
RUN=max([p for p in (ROOT/"observed_print_classifier_v8_6_notebook46").iterdir() if p.is_dir()], key=lambda p:p.stat().st_mtime)
TABLES=RUN/"tables"
whole=pd.read_csv(TABLES/"whole_page_candidate_metrics.csv")
print(RUN)


## Global scatter by restorer


In [ ]:
plt.figure(figsize=(10,8))
for r,g in whole.groupby("restorer"):
 plt.scatter(g.false_inserted_ink,g.missing_ink,label=r,alpha=.5)
plt.xlabel("False inserted ink")
plt.ylabel("Missing ink")
plt.legend(); plt.grid(); plt.show()


## Faceted by restorer and source group


In [ ]:
g=sns.FacetGrid(whole,col="restorer",col_wrap=2,hue="source_group",height=5)
g.map_dataframe(sns.scatterplot,x="false_inserted_ink",y="missing_ink")
g.add_legend(); plt.show()


## Pareto frontier overlay


In [ ]:
summary=whole.groupby("restorer",as_index=False).agg(false_inserted=("false_inserted_ink","mean"),missing=("missing_ink","mean"))
pts=summary[["false_inserted","missing"]].sort_values("false_inserted")
plt.figure(figsize=(8,6))
plt.scatter(summary.false_inserted,summary.missing,s=200)
for _,r in summary.iterrows(): plt.annotate(r.restorer,(r.false_inserted,r.missing))
plt.plot(pts.false_inserted,pts.missing,"k--")
plt.xlabel("Mean false inserted")
plt.ylabel("Mean missing")
plt.title("Pareto trade-off")
plt.grid(); plt.show()


## Per-source-group Pareto frontiers


In [ ]:
rows=[]
for sg,g in whole.groupby("source_group"):
 s=g.groupby("restorer",as_index=False).agg(false_inserted=("false_inserted_ink","mean"),missing=("missing_ink","mean"))
 s["source_group"]=sg
 rows.append(s)
pareto_by_group=pd.concat(rows)
display(pareto_by_group)


## Variance and stability


In [ ]:
variance=whole.groupby("restorer").agg(false_mean=("false_inserted_ink","mean"),false_sd=("false_inserted_ink","std"),missing_mean=("missing_ink","mean"),missing_sd=("missing_ink","std"))
variance["false_cv"]=variance.false_sd/variance.false_mean
variance["missing_cv"]=variance.missing_sd/variance.missing_mean
display(variance.sort_values("missing_cv"))


## Mahalanobis outlier detection


In [ ]:
out=[]
for r,g in whole.groupby("restorer"):
 X=g[["false_inserted_ink","missing_ink"]].values
 if len(X)<3: continue
 VI=np.linalg.pinv(np.cov(X.T))
 c=X.mean(axis=0)
 d=[mahalanobis(v,c,VI) for v in X]
 tmp=g.copy(); tmp["mahalanobis"]=d
 out.append(tmp.sort_values("mahalanobis",ascending=False).head(10))
outliers=pd.concat(out)
display(outliers[["page","source_group","restorer","false_inserted_ink","missing_ink","mahalanobis"]])


## Worst pages by restorer


In [ ]:
for r,g in whole.groupby("restorer"):
 print("\n",r)
 display(g.sort_values("missing_ink",ascending=False).head(10)[["page","source_group","missing_ink","false_inserted_ink"]])


## Export analysis


In [ ]:
OUT=RUN/"notebook47R1_tradeoff_analysis"
OUT.mkdir(exist_ok=True)
variance.to_csv(OUT/"variance_summary.csv")
pareto_by_group.to_csv(OUT/"pareto_by_source_group.csv",index=False)
outliers.to_csv(OUT/"outlier_pages.csv",index=False)
print("Written:",OUT)
